In [10]:
# Load, Merge, and Add Advanced Features 

import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# 1. Load Data
stops_df = pd.read_csv('../data/raw/stops.txt')
routes_df = pd.read_csv('../data/raw/routes.txt')
trips_df = pd.read_csv('../data/raw/trips.txt')
stop_times_df = pd.read_csv('../data/raw/stop_times.txt')

# Merge
trip_route_df = pd.merge(trips_df, routes_df, on='route_id', how='left')
schedule_df = pd.merge(stop_times_df, trip_route_df, on='trip_id', how='left')
full_network_df = pd.merge(schedule_df, stops_df, on='stop_id', how='left')

# 2. Time Conversion
def time_to_seconds(time_str):
    if pd.isna(time_str): return np.nan
    h, m, s = map(int, time_str.split(':'))
    return h * 3600 + m * 60 + s

full_network_df['scheduled_seconds'] = full_network_df['arrival_time'].apply(time_to_seconds)
full_network_df = full_network_df.dropna(subset=['scheduled_seconds']).copy()

# 3. Create Basic Time Features
full_network_df['hour'] = (full_network_df['scheduled_seconds'] // 3600) % 24
full_network_df['is_peak'] = full_network_df['hour'].apply(lambda x: 1 if (8 <= x <= 10) or (17 <= x <= 20) else 0)

# 4. Advanced Features (Upgrades)
print("Adding advanced features...")

# Task 2: Continuous Station Congestion
congestion_weights = {
    'Ameerpet': 3.5,
    'MG Bus Station': 3.0,
    'Parade Ground': 2.5,
    'Raidurg': 2.0, 
    'Secunderabad East': 2.0
}
full_network_df['station_congestion'] = full_network_df['stop_name'].map(congestion_weights).fillna(1.0)

# Task 3: Route Encoding
if 'route_id' in full_network_df.columns:
    route_encoder = LabelEncoder()
    full_network_df['route_encoded'] = route_encoder.fit_transform(full_network_df['route_id'].astype(str))
    joblib.dump(route_encoder, '../models/route_encoder.pkl')

# Task 4: Day of Week simulation
np.random.seed(42)
full_network_df['day_of_week'] = np.random.randint(0, 7, size=len(full_network_df))
full_network_df['is_weekend'] = full_network_df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

print("Base features loaded successfully.")

Adding advanced features...
Base features loaded successfully.


In [11]:
# Simulate Refined Delays & Calculate Previous Delay

print("Simulating delays with sequence variation...")
np.random.seed(42)

# Crucial: Sort by trip and sequence for proper delay propagation
full_network_df = full_network_df.sort_values(by=['trip_id', 'stop_sequence'])

delay_records = []
current_trip = None
cumulative_delay = 0

for index, row in full_network_df.iterrows():
    if row['trip_id'] != current_trip:
        current_trip = row['trip_id']
        cumulative_delay = np.random.exponential(scale=0.5) # Base start noise
        
    # Propagation: Variation based on sequence (Later stops have higher risk)
    sequence_penalty = (row['stop_sequence'] * 0.05) 
    cumulative_delay += np.random.uniform(0, 0.3 + sequence_penalty) 
    
    station_delay = cumulative_delay
    
    # Peak hour penalty
    if row['is_peak'] == 1:
        station_delay += np.random.uniform(1, 3)
        
    # Station Congestion Multiplier (Replaces binary penalty)
    station_delay += np.random.uniform(0.5, 1.5) * row['station_congestion']
        
    # Small random chance to recover/be on time
    if np.random.rand() < 0.10:
        station_delay = max(0, station_delay - np.random.uniform(1, 3))
        cumulative_delay = station_delay 
        
    delay_records.append(station_delay)

full_network_df['delay_minutes'] = delay_records
full_network_df['delay_minutes'] = full_network_df['delay_minutes'].clip(lower=0, upper=40).round(2)
print("Delays generated!")

# Task 1: Calculate Previous Delay
print("Calculating previous stop delays...")
full_network_df['prev_delay'] = full_network_df.groupby('trip_id')['delay_minutes'].shift(1).fillna(0.0)
print("Previous delays added!")

Simulating delays with sequence variation...
Delays generated!
Calculating previous stop delays...
Previous delays added!


In [12]:
# Clean, Encode, Validate, Split, and Save

# 1. Validation & Cleaning
print(f"Rows before cleaning: {len(full_network_df)}")
full_network_df = full_network_df.drop_duplicates()
full_network_df = full_network_df.dropna(subset=['stop_id', 'delay_minutes'])
print(f"Rows after cleaning: {len(full_network_df)}")

# 2. Station ID Encoding
label_encoder = LabelEncoder()
full_network_df['station_id_encoded'] = label_encoder.fit_transform(full_network_df['stop_id'])
joblib.dump(label_encoder, '../models/station_id_encoder.pkl')

# 3. Final Feature Selection (Updated List)
features = [
    'station_id_encoded', 'hour', 'is_peak', 'station_congestion', 
    'stop_sequence', 'prev_delay', 'route_encoded', 
    'day_of_week', 'is_weekend'
]
target = 'delay_minutes'

# Ensure final dataframe is pristine
df_final = full_network_df[features + [target]].dropna().copy()
print(f"Rows after final NaN drop: {len(df_final)}")

X = df_final[features]
y = df_final[target]

# 4. Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# 5. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, '../models/feature_scaler.pkl')
print("Scaler saved to models/feature_scaler.pkl")

# 6. Save the final pristine dataset
final_ml_df = X.copy()
final_ml_df['delay_minutes'] = y
final_ml_df.to_csv('../data/processed/full_network_ml_upgraded.csv', index=False)
print("Final dataset saved to data/processed/full_network_ml_upgraded.csv")

print("\n--- Summary Statistics ---")
print(final_ml_df[['delay_minutes', 'prev_delay', 'station_congestion']].describe().round(2))

print("\n--- Upgraded Pipeline Complete! Ready for Phase 3 ---")
print(final_ml_df.head())

Rows before cleaning: 61037
Rows after cleaning: 61037
Rows after final NaN drop: 61037
Training data shape: (48829, 9)
Testing data shape: (12208, 9)
Scaler saved to models/feature_scaler.pkl
Final dataset saved to data/processed/full_network_ml_upgraded.csv

--- Summary Statistics ---
       delay_minutes  prev_delay  station_congestion
count       61037.00    61037.00            61037.00
mean            7.22        6.64                1.16
std             4.71        4.63                0.53
min             0.00        0.00                1.00
25%             3.52        3.08                1.00
50%             6.20        5.70                1.00
75%            10.08        9.47                1.00
max            29.71       28.52                3.50

--- Upgraded Pipeline Complete! Ready for Phase 3 ---
   station_id_encoded  hour  is_peak  station_congestion  stop_sequence  \
0                  56     6        0                 1.0              1   
1                 107     6   